# Sentient AI — Milestone 1 Resubmission Notebook
INAI 6986 · Foluwasade Ajagbe

This notebook rebuilds the data preparation, baseline, comparator, and candidate models
from the raw MTSamples source, and adds the LLM (Claude API) candidate.

**Note on Baseline:** the original `train.csv`/`validation.csv` from the first Milestone 1
submission were not recovered, so this notebook regenerates the split from
`mtsamples-v3.csv` with a fresh, fully-documented set of rules below. Every number in
this notebook is produced by the code in this notebook — nothing is pasted in.

In [ ]:
import hashlib, re, json, warnings
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report



SRC = "mtsamples-v3.csv"   # place the raw source file next to this notebook
SEED = 42

## 1. Source hashing and load

In [ ]:
with open(SRC, "rb") as f:
    src_hash = hashlib.sha256(f.read()).hexdigest()
print("SHA-256 of source:", src_hash)

df = pd.read_csv(SRC, on_bad_lines='skip', engine='python').rename(columns={"Unnamed: 0": "row_id"})
for col in ["medical_specialty", "sample_name", "transcription", "keywords", "description"]:
  df[col] = df[col].astype(str).str.strip()
print("Raw rows:", len(df))

SHA-256 of source: baa6d482c2d25362f6e082fda5756a8faaef805e2cf6b03fd5c6f7946588b272
Raw rows: 4999


## 2. Class selection

**Rule:** keep single-specialty outpatient referral targets — the kind of specialty a
GP would route a non-emergency patient to. Exclude document-type / encounter-type
categories (`Surgery`, `Consult - History and Phy.`, `SOAP / Chart / Progress Notes`,
`Discharge Summary`, `Emergency Room Reports`, `Office Notes`, `Letters`, `Autopsy`,
`General Medicine`) because those describe *where a note was written*, not a specialty
a referral engine would route to.

In [ ]:
#Target class and select a specialist
TARGET_CLASSES = [
    "Cardiovascular / Pulmonary", "Orthopedic", "Gastroenterology", "Neurology",
    "Obstetrics / Gynecology", "Urology", "ENT - Otolaryngology", "Nephrology",
    "Dermatology",
]
sub = df[df["medical_specialty"].isin(TARGET_CLASSES)].copy()
sub = sub[~sub["transcription"].isin(["", "nan"])].copy()
print("After class selection + drop-empty:", len(sub))

After class selection + drop-empty: 1690


## 3. Exact-duplicate removal

In [ ]:
#Remove the empty duplicate before splitting
before = len(sub)
sub = sub.drop_duplicates(subset="transcription", keep="first").copy()
print(f"After exact-duplicate removal: {len(sub)} (dropped {before - len(sub)})")

After exact-duplicate removal: 1610 (dropped 80)


## 4. Multi-label removal

**Rule:** a note is flagged multi-label if its `keywords` field name-checks one of the
*other* 8 target specialties (lowercased substring match on each specialty's name
tokens). This catches notes MTSamples' own indexers associated with more than one
target specialty.

In [ ]:
def specialty_tokens(name):
    parts = re.split(r"[\/\-]", name)
    return [p.strip().lower() for p in parts if len(p.strip()) > 3]

spec_tokens = {c: specialty_tokens(c) for c in TARGET_CLASSES}

def is_multilabel(row):
    kw = str(row["keywords"]).lower()
    own = row["medical_specialty"]
    return any(any(tok in kw for tok in toks) for other, toks in spec_tokens.items() if other != own)

# Only proceed if 'sub' DataFrame is not empty
if len(sub) > 0:
    sub["multilabel_flag"] = sub.apply(is_multilabel, axis=1)
    before = len(sub)
    sub = sub[~sub["multilabel_flag"]].copy()
    print(f"After multi-label removal: {len(sub)} (dropped {before - len(sub)})")
    # Only print value counts if 'medical_specialty' column exists
    if "medical_specialty" in sub.columns:
        print(sub["medical_specialty"].value_counts())
    else:
        print("DataFrame 'sub' is empty or 'medical_specialty' column is missing after filtering.")
else:
    print("DataFrame 'sub' is empty, skipping multi-label removal and value counts.")

After multi-label removal: 1470 (dropped 140)
medical_specialty
Cardiovascular / Pulmonary    359
Orthopedic                    353
Gastroenterology              203
Urology                       156
Obstetrics / Gynecology       154
ENT - Otolaryngology           95
Nephrology                     67
Neurology                      54
Dermatology                    29
Name: count, dtype: int64


## 5. Pre-decision truncation (the leakage fix)

Truncate every note at the **earliest** occurrence of a post-decision section header
(`PREOPERATIVE DIAGNOSIS`, `IMPRESSION`, `ASSESSMENT`, `PLAN`, `PROCEDURE`, etc.).
Notes with less than 40 characters of pre-decision text remaining are dropped — mostly
short operative notes that open directly with `PREOPERATIVE DIAGNOSIS:` and have no
usable referral-style narrative before it.

In [ ]:
DECISION_HEADERS = [
    "PREOPERATIVE DIAGNOSIS", "PREOPERATIVE DIAGNOSES", "POSTOPERATIVE DIAGNOSIS",
    "POSTOPERATIVE DIAGNOSES", "PROCEDURE PERFORMED", "PROCEDURE IN DETAIL",
    "DESCRIPTION OF PROCEDURE", "PROCEDURE", "PROCEDURES", "ANESTHESIA", "FINDINGS",
    "IMPRESSION", "ASSESSMENT AND PLAN", "ASSESSMENT", "DIAGNOSIS", "DIAGNOSES", "PLAN",
    "COMPLICATIONS", "ESTIMATED BLOOD LOSS", "SPECIMENS", "INDICATIONS", "INDICATION",
    "DISPOSITION", "RECOMMENDATION", "RECOMMENDATIONS",
]
pattern = re.compile(
    r"(?:^|[\.,\n])\s*(" + "|".join(re.escape(h) for h in DECISION_HEADERS) + r")\s*:",
    re.IGNORECASE,
)

def truncate_pre_decision(text):
    text = str(text)
    m = pattern.search(text)
    if m:
        return text[:m.start()].strip(), True, m.group(1).upper()
    return text.strip(), False, None

trunc_results = sub["transcription"].apply(truncate_pre_decision)
sub["text"] = trunc_results.apply(lambda x: x[0])
sub["truncated"] = trunc_results.apply(lambda x: x[1])
sub["cut_header"] = trunc_results.apply(lambda x: x[2])
sub["text_len"] = sub["text"].str.len()

print("Truncated (decision header found):", sub["truncated"].sum(), "/", len(sub))

before = len(sub)
sub = sub[sub["text_len"] >= 40].copy()
print(f"After dropping too-short pre-decision text: {len(sub)} (dropped {before - len(sub)})")
print(sub["medical_specialty"].value_counts())

Truncated (decision header found): 1236 / 1470
After dropping too-short pre-decision text: 741 (dropped 729)
medical_specialty
Cardiovascular / Pulmonary    230
Orthopedic                    147
Gastroenterology               78
Obstetrics / Gynecology        70
Urology                        59
Nephrology                     52
Neurology                      51
ENT - Otolaryngology           37
Dermatology                    17
Name: count, dtype: int64


## Manual audit of untruncated notes immediately after Section 5, before near-duplicate grouping and splitting.

In [ ]:
# Select notes where the truncation rule found no recognized header.
untruncated = sub.loc[~sub["truncated"]].copy()

print("Untruncated notes available:", len(untruncated))

# Preserve the source identifier.
if "Record ID" in untruncated.columns:
    untruncated["audit_id"] = untruncated["Record ID"].astype(str)
elif "row_id" in untruncated.columns:
    untruncated["audit_id"] = untruncated["row_id"].astype(str)
else:
    untruncated["audit_id"] = untruncated.index.astype(str)

audit = untruncated.sample(
    n=min(20, len(untruncated)),
    random_state=SEED
)[["audit_id", "medical_specialty", "transcription"]].copy()

# Fill these columns after reading each note.
audit["decision_content_present"] = ""  # yes / no / uncertain
audit["evidence"] = ""                  # exact excerpt from this note
audit["missed_heading_or_phrase"] = ""
audit["proposed_action"] = ""           # keep / truncate / exclude / review
audit["reviewer_notes"] = ""

audit.to_csv("untruncated_notes_audit_20.csv", index=False)

# Print full notes so long text is not hidden.
for number, row in enumerate(audit.itertuples(index=False), start=1):
    print(f"\nNOTE {number} | ID: {row.audit_id}")
    print(f"Source specialty: {row.medical_specialty}")
    print(row.transcription)
    print("-" * 80)

Untruncated notes available: 231

NOTE 1 | ID: 4897
Source specialty: Cardiovascular / Pulmonary
Please accept this letter of follow up on patient xxx xxx. He is now three months out from a left carotid angioplasty and stent placement. He was a part of a CapSure trial. He has done quite well, with no neurologic or cardiac event in the three months of follow up. He had a follow-up ultrasound performed today that shows the stent to be patent, with no evidence of significant recurrence.,Sincerely,,XYZ, MD,
--------------------------------------------------------------------------------

NOTE 2 | ID: 2552
Source specialty: Obstetrics / Gynecology
Pap smear in November 2006 showed atypical squamous cells of undetermined significance.  She has a history of an abnormal Pap smear.  At that time, she was diagnosed with CIN 3 as well as vulvar intraepithelial neoplasia.  She underwent a cone biopsy that per her report was negative for any pathology.  She had no vulvar treatment at that time.  Si

## 6. Near-duplicate check

Cosine similarity on TF-IDF vectors of the *pre-decision* text, between all pairs of
remaining notes. Pairs at or above 0.9 are grouped with union-find so a near-duplicate
pair can never be split across train and validation.

In [ ]:
import re
import numpy as np
import pandas as pd
from dataclasses import dataclass
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedGroupKFold

# --------------------------------------------------------------------------
# 1. Direct Implementation of Near-Duplicates Deduplication System
# --------------------------------------------------------------------------

@dataclass
class DedupConfig:
    text_col: str
    label_col: str
    id_col: str
    duplicate_threshold: float = 0.90
    review_threshold: float = 0.75
    min_chars: int = 40
    mask_variables: bool = True
    random_state: int = 42

@dataclass
class DedupReport:
    n_too_short: int
    n_exact_duplicates: int
    n_near_duplicates: int
    n_clusters: int

    def to_json(self, path):
        import json
        with open(path, 'w') as f:
            json.dump(self.__dict__, f, indent=2)

def normalise(text: str, mask_variables: bool = True) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    if mask_variables:
        text = re.sub(r"\b\d+\s*-\s*(?:year|yr)\s*-\s*old\b", "<age>", text, flags=re.IGNORECASE)
        text = re.sub(r"\b\d+\s+year\s+old\b", "<age>", text, flags=re.IGNORECASE)
        text = re.sub(r"\b\d{2}/\d{2}/\d{4}\b", "<date>", text)
        text = re.sub(r"\b\d{5,}\b", "<num>", text)
    return text

def check_dataframe(df: pd.DataFrame, cfg: DedupConfig):
    problems = []
    for col in [cfg.text_col, cfg.label_col, cfg.id_col]:
        if col not in df.columns:
            problems.append(f"Missing column: {col}")
    return len(problems) == 0, problems

def calibrate(df: pd.DataFrame, cfg: DedupConfig, n_show: int = 10):
    print("Calibrating similarity distributions...")
    processed = df[cfg.text_col].apply(lambda x: normalise(x, cfg.mask_variables))
    vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
    tfidf_matrix = vectorizer.fit_transform(processed)
    sim_matrix = cosine_similarity(tfidf_matrix)

    sims = []
    n = len(df)
    for i in range(n):
        for j in range(i + 1, n):
            sims.append(sim_matrix[i, j])
    return sims, sim_matrix

def find_near_duplicates(df: pd.DataFrame, cfg: DedupConfig):
    lengths = df[cfg.text_col].astype(str).str.len()
    too_short_mask = lengths < cfg.min_chars
    n_too_short = too_short_mask.sum()

    processed_texts = df[cfg.text_col].apply(lambda x: normalise(x, mask_variables=cfg.mask_variables))
    vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
    tfidf_matrix = vectorizer.fit_transform(processed_texts)
    sim_matrix = cosine_similarity(tfidf_matrix)

    n = len(df)
    parent = list(range(n))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_i] = root_j

    pairs_list = []
    for i in range(n):
        for j in range(i + 1, n):
            if too_short_mask.iloc[i] or too_short_mask.iloc[j]:
                continue
            sim = sim_matrix[i, j]
            band = None
            if sim >= cfg.duplicate_threshold:
                union(i, j)
                band = "duplicate"
            elif sim >= cfg.review_threshold:
                band = "review"
            if band:
                pairs_list.append({"id1": df.iloc[i][cfg.id_col], "id2": df.iloc[j][cfg.id_col], "sim": sim, "band": band})

    root_to_cluster = {}
    cluster_ids = []
    cluster_counter = 0
    for i in range(n):
        if too_short_mask.iloc[i]:
            cluster_ids.append(None)
            continue
        root = find(i)
        if root not in root_to_cluster:
            root_to_cluster[root] = f"C_{cluster_counter}"
            cluster_counter += 1
        cluster_ids.append(root_to_cluster[root])

    df_out = df.copy()
    df_out["cluster_id"] = cluster_ids
    null_mask = df_out["cluster_id"].isna() & (~too_short_mask)
    for idx in df_out[null_mask].index:
        df_out.loc[idx, "cluster_id"] = f"C_{cluster_counter}"
        cluster_counter += 1

    pairs_df = pd.DataFrame(pairs_list)
    report = DedupReport(
        n_too_short=int(n_too_short),
        n_exact_duplicates=int((sim_matrix >= 0.999).sum() - n) // 2,
        n_near_duplicates=len(pairs_df[pairs_df.band == "duplicate"]),
        n_clusters=cluster_counter
    )
    return df_out, pairs_df, report

def grouped_split(df: pd.DataFrame, cfg: DedupConfig, test_size: float = 0.20, val_size: float = 0.15):
    df_split = df.copy()
    # Use stratified group split to avoid class and cluster leakage
    group_rep = df_split.groupby("cluster_id").first().reset_index()
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=cfg.random_state)

    # Generate a deterministic train / val / test mapping
    train_idx, test_idx = next(sgkf.split(group_rep, group_rep[cfg.label_col], groups=group_rep["cluster_id"]))
    test_clusters = set(group_rep.iloc[test_idx]["cluster_id"])
    remaining_clusters = group_rep.iloc[train_idx]

    sgkf_val = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=cfg.random_state)
    train_sub_idx, val_idx = next(sgkf_val.split(remaining_clusters, remaining_clusters[cfg.label_col], groups=remaining_clusters["cluster_id"]))
    val_clusters = set(remaining_clusters.iloc[val_idx]["cluster_id"])

    splits = []
    for cid in df_split["cluster_id"]:
        if cid in test_clusters:
            splits.append("test")
        elif cid in val_clusters:
            splits.append("validation")
        else:
            splits.append("train")
    df_split["split"] = splits
    return df_split

def audit_split(df: pd.DataFrame, pairs: pd.DataFrame, cfg: DedupConfig):
    cross_split_duplicates = 0
    id_to_split = df.set_index(cfg.id_col)["split"].to_dict()
    for _, row in pairs.iterrows():
        if row["band"] == "duplicate":
            s1 = id_to_split.get(row["id1"])
            s2 = id_to_split.get(row["id2"])
            if s1 and s2 and s1 != s2:
                cross_split_duplicates += 1
    return {
        "cross_split_duplicate_pairs": cross_split_duplicates,
        "split_sizes": df["split"].value_counts().to_dict()
    }

# --------------------------------------------------------------------------
# 2. Re-running the deduplication pipeline using our local classes
# --------------------------------------------------------------------------

# Define configuration matching our pre-decision text column ('text' in 'sub')
cfg = DedupConfig(
    text_col="text",
    label_col="medical_specialty",
    id_col="Record ID",
    duplicate_threshold=0.90,
    review_threshold=0.75,
    min_chars=40,
    mask_variables=True,
    random_state=SEED
)

# Validate and run preflight checks
ok, problems = check_dataframe(sub, cfg)
if not ok:
    print("Preflight validation failed:", problems)
else:
    # Run calibration, duplicate grouping, and split audit
    sims, S = calibrate(sub, cfg)
    sub_clustered, pairs, report = find_near_duplicates(sub, cfg)
    print(report)

    pairs[pairs.band == "review"].to_csv("review_band_pairs.csv", index=False)
    print(f"{len(pairs[pairs.band=='review'])} pairs exported for adjudication")

    # Conduct group-aware train/test/val partitioning
    sub_clustered = grouped_split(sub_clustered, cfg, test_size=0.20, val_size=0.15)
    audit = audit_split(sub_clustered, pairs, cfg)
    print("cross-split duplicate pairs:", audit["cross_split_duplicate_pairs"])
    print("split sizes:", audit["split_sizes"])


Calibrating similarity distributions...
DedupReport(n_too_short=0, n_exact_duplicates=0, n_near_duplicates=1, n_clusters=740)
20 pairs exported for adjudication
cross-split duplicate pairs: 0
split sizes: {'train': 444, 'validation': 149, 'test': 148}


## 7. Stratified, group-aware train/validation/ test split

17% validation, stratified by specialty, with near-duplicate groups kept intact.

In [ ]:
from sklearn.model_selection import train_test_split

# Configure and run the near-duplicate detection to define `sub_clustered`
cfg = DedupConfig(
    text_col="text",
    label_col="medical_specialty",
    id_col="Record ID",
    duplicate_threshold=0.90,
    review_threshold=0.75,
    min_chars=40,
    mask_variables=True,
    random_state=SEED
)

# Run the deduplication system to obtain sub_clustered
sub_clustered, _, _ = find_near_duplicates(sub, cfg)

# Map "Record ID" to "case_id" so the downstream code has the expected case_id column
if "case_id" not in sub_clustered.columns and "Record ID" in sub_clustered.columns:
    sub_clustered["case_id"] = "case_" + sub_clustered["Record ID"].astype(str)

# Filter out classes with fewer than 2 members to allow stratified splitting
class_counts = sub_clustered["medical_specialty"].value_counts()
valid_classes = class_counts[class_counts >= 2].index
sub_clustered = sub_clustered[sub_clustered["medical_specialty"].isin(valid_classes)].copy()

# Each near-duplicate group must have exactly one specialty.
assert sub_clustered.groupby("cluster_id")["medical_specialty"].nunique().eq(1).all()

group_rep = sub_clustered.groupby("cluster_id").first().reset_index()

# Reproduce the existing train/validation split.
original_train_groups, val_groups = train_test_split(
    group_rep["cluster_id"],
    test_size=0.17,
    stratify=group_rep["medical_specialty"],
    random_state=SEED,
)

# Reserve about 18% of the old training groups as the new test set.
# This gives approximately 68% train / 17% validation / 15% test overall.
original_train_rep = group_rep[
    group_rep["cluster_id"].isin(original_train_groups)
]
train_groups, test_groups = train_test_split(
    original_train_rep["cluster_id"],
    test_size=0.18,
    stratify=original_train_rep["medical_specialty"],
    random_state=SEED,
)

train_df = sub_clustered[sub_clustered["cluster_id"].isin(train_groups)].copy()
val_df = sub_clustered[sub_clustered["cluster_id"].isin(val_groups)].copy()
test_df = sub_clustered[sub_clustered["cluster_id"].isin(test_groups)].copy()

out_cols = [
    "case_id", "medical_specialty", "text",
    "sample_name", "truncated", "cut_header"
]

def make_split(df):
    return df[out_cols].rename(
        columns={"medical_specialty": "specialty"}
    )

train = make_split(train_df)
val = make_split(val_df)
test = make_split(test_df)

# Check that cases and near-duplicate groups never cross partitions.
assert set(train.case_id).isdisjoint(val.case_id)
assert set(train.case_id).isdisjoint(test.case_id)
assert set(val.case_id).isdisjoint(test.case_id)

assert set(train_df.cluster_id).isdisjoint(val_df.cluster_id)
assert set(train_df.cluster_id).isdisjoint(test_df.cluster_id)
assert set(val_df.cluster_id).isdisjoint(test_df.cluster_id)

assert all(df.text.notna().all() for df in (train, val, test))
assert len(train) + len(val) + len(test) == len(sub_clustered)
assert set(train.specialty) == set(val.specialty) == set(test.specialty)

train.to_csv("train.csv", index=False)
val.to_csv("validation.csv", index=False)
test.to_csv("test.csv", index=False)

# Save the fixed group assignment for reproducibility.
split_manifest = sub_clustered[["case_id", "cluster_id"]].copy()
split_manifest["split"] = "train"
split_manifest.loc[
    split_manifest["cluster_id"].isin(val_groups), "split"
] = "validation"
split_manifest.loc[
    split_manifest["cluster_id"].isin(test_groups), "split"
] = "test"
split_manifest.to_csv("split_manifest.csv", index=False)

print("Train:", len(train), "Validation:", len(val), "Test:", len(test))
print("\nTrain class counts:\n", train.specialty.value_counts())
print("\nValidation class counts:\n", val.specialty.value_counts())
print("\nTest class counts:\n", test.specialty.value_counts())

labels = sorted(train.specialty.unique().tolist())

Train: 504 Validation: 126 Test: 111

Train class counts:
 specialty
Cardiovascular / Pulmonary    157
Orthopedic                    100
Gastroenterology               53
Obstetrics / Gynecology        48
Urology                        40
Nephrology                     35
Neurology                      34
ENT - Otolaryngology           25
Dermatology                    12
Name: count, dtype: int64

Validation class counts:
 specialty
Cardiovascular / Pulmonary    39
Orthopedic                    25
Gastroenterology              13
Obstetrics / Gynecology       12
Urology                       10
Neurology                      9
Nephrology                     9
ENT - Otolaryngology           6
Dermatology                    3
Name: count, dtype: int64

Test class counts:
 specialty
Cardiovascular / Pulmonary    34
Orthopedic                    22
Gastroenterology              12
Obstetrics / Gynecology       10
Urology                        9
Neurology                      8
Nephrology

## 8. Leakage check: truncated vs. untruncated text

Same split, same model, run once on the pre-decision text and once on the full
(untruncated) transcription, to show what the post-decision sections were worth.

In [ ]:
full_lookup = sub_clustered.set_index("case_id")["transcription"]
train_full = train["case_id"].map(full_lookup)
val_full = val["case_id"].map(full_lookup)

tfidf_leak = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english")
Xtr_full = tfidf_leak.fit_transform(train_full)
Xval_full = tfidf_leak.transform(val_full)
clf_leak = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr_full, train["specialty"])
pred_leak = clf_leak.predict(Xval_full)

acc_leak = accuracy_score(val["specialty"], pred_leak)
f1_leak = f1_score(val["specialty"], pred_leak, average="macro", zero_division=0)
print(f"Logistic Regression on UNTRUNCATED text: accuracy={acc_leak:.3f}  macro F1={f1_leak:.3f}")
print("(kept here to demonstrate the leakage effect — the truncated-text score below is the real one)")

Logistic Regression on UNTRUNCATED text: accuracy=0.873  macro F1=0.885
(kept here to demonstrate the leakage effect — the truncated-text score below is the real one)


## 9. Keyword baseline

**Baseline:** every specialty's keyword list was written by reading a sample of
**training** notes only for that specialty. No validation note was consulted while
writing this list.

In [ ]:
KEYWORDS = {
    "Cardiovascular / Pulmonary": ["heart", "cardiac", "coronary", "pulmonary", "lung", "chest pain", "ejection fraction", "artery"],
    "Orthopedic": ["fracture", "joint", "knee", "hip", "shoulder", "orthopedic", "bone", "ligament"],
    "Gastroenterology": ["abdomen", "abdominal", "colon", "bowel", "gastric", "liver", "gi ", "stomach"],
    "Obstetrics / Gynecology": ["pregnan", "uterus", "cervix", "ovary", "vaginal", "gestational", "fetal"],
    "Urology": ["bladder", "prostate", "urinary", "kidney stone", "urology", "urethra", "scrotal"],
    "Nephrology": ["renal", "kidney", "dialysis", "nephro", "creatinine", "glomerular"],
    "Neurology": ["seizure", "neurolog", "headache", "brain", "cranial", "stroke", "weakness"],
    "ENT - Otolaryngology": ["throat", "ear", "nasal", "sinus", "tonsil", "hearing", "larynx"],
    "Dermatology": ["skin", "rash", "lesion", "dermat", "mole", "biopsy of the skin"],
}
def keyword_predict(text):
    t = str(text).lower()
    scores = {c: sum(t.count(k) for k in kws) for c, kws in KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "Cardiovascular / Pulmonary"

def report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    print(f"\n=== {name} ===\nTop-1 accuracy: {acc:.3f}   Macro F1: {f1:.3f}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    return acc, f1

dummy = DummyClassifier(strategy="most_frequent").fit(train["text"], train["specialty"])
report("Majority-class sanity check", val["specialty"], dummy.predict(val["text"]))

kw_pred = val["text"].apply(keyword_predict)
report("Keyword baseline", val["specialty"], kw_pred)


=== Majority-class sanity check ===
Top-1 accuracy: 0.310   Macro F1: 0.053
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.31      1.00      0.47        39
               Dermatology       0.00      0.00      0.00         3
      ENT - Otolaryngology       0.00      0.00      0.00         6
          Gastroenterology       0.00      0.00      0.00        13
                Nephrology       0.00      0.00      0.00         9
                 Neurology       0.00      0.00      0.00         9
   Obstetrics / Gynecology       0.00      0.00      0.00        12
                Orthopedic       0.00      0.00      0.00        25
                   Urology       0.00      0.00      0.00        10

                  accuracy                           0.31       126
                 macro avg       0.03      0.11      0.05       126
              weighted avg       0.10      0.31      0.15       126


=== Keyword baseline ===
Top-1 accu

(0.5793650793650794, 0.5402702952895921)

## 10. Non-DL comparator: Logistic Regression on TF-IDF

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english")
Xtr = tfidf.fit_transform(train["text"])
Xval = tfidf.transform(val["text"])

logreg = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, train["specialty"])
report("Logistic Regression (TF-IDF)", val["specialty"], logreg.predict(Xval))


=== Logistic Regression (TF-IDF) ===
Top-1 accuracy: 0.849   Macro F1: 0.843
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.88      0.90      0.89        39
               Dermatology       1.00      1.00      1.00         3
      ENT - Otolaryngology       1.00      0.67      0.80         6
          Gastroenterology       0.67      0.77      0.71        13
                Nephrology       0.80      0.89      0.84         9
                 Neurology       0.80      0.89      0.84         9
   Obstetrics / Gynecology       0.92      0.92      0.92        12
                Orthopedic       0.88      0.88      0.88        25
                   Urology       0.86      0.60      0.71        10

                  accuracy                           0.85       126
                 macro avg       0.87      0.83      0.84       126
              weighted avg       0.86      0.85      0.85       126



(0.8492063492063492, 0.8430134677307151)

## 11. MLP candidates — fixed

Previous run showed MLP 64-32 collapsing to near-majority-class behavior. Fix: explicit
`max_iter=3000`, `early_stopping=False` (401-row train set is too small to spend 10% of
it on an internal holdout), and convergence is checked and printed, not assumed.

In [ ]:
def fit_mlp(hidden, tag):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        clf = MLPClassifier(hidden_layer_sizes=hidden, max_iter=3000,
                             early_stopping=False, random_state=SEED, alpha=1e-3)
        clf.fit(Xtr, train["specialty"])
        conv_warnings = [str(x.message) for x in w if "converge" in str(x.message).lower()]
    print(f"[{tag}] iterations run: {clf.n_iter_}  final loss: {clf.loss_:.4f}  convergence warnings: {len(conv_warnings)}")
    report(f"MLP {tag} (TF-IDF)", val["specialty"], clf.predict(Xval))
    return clf

mlp_64 = fit_mlp((64,), "64")
mlp_64_32 = fit_mlp((64, 32), "64-32")

[64] iterations run: 186  final loss: 0.0149  convergence warnings: 0

=== MLP 64 (TF-IDF) ===
Top-1 accuracy: 0.849   Macro F1: 0.817
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.79      0.97      0.87        39
               Dermatology       1.00      0.67      0.80         3
      ENT - Otolaryngology       1.00      0.50      0.67         6
          Gastroenterology       0.75      0.69      0.72        13
                Nephrology       0.88      0.78      0.82         9
                 Neurology       1.00      0.89      0.94         9
   Obstetrics / Gynecology       0.83      0.83      0.83        12
                Orthopedic       0.92      0.92      0.92        25
                   Urology       0.88      0.70      0.78        10

                  accuracy                           0.85       126
                 macro avg       0.89      0.77      0.82       126
              weighted avg       0.86      0.85

## 12. LLM candidate — Claude API

This is the declared model family and was never tested in the first submission.
Zero-shot, same 126 validation cases, same pre-decision-truncated text as every other
candidate above.

**Update:** This candidate can be executed successfully on the validation set using Claude 3.5 Sonnet, demonstrating a peak performance of **accuracy and Macro F1-score**.

In [17]:
import os, json
import pandas as pd
from getpass import getpass
!pip install anthropic
from anthropic import APIError

# Fallback to load the validation dataset if 'val' is missing from global memory
try:
    _ = val
except NameError:
    print("Validation DataFrame 'val' is missing from memory. Loading 'validation.csv'...")
    val = pd.read_csv("validation.csv")

# Ensure labels are defined based on the validation set
try:
    labels = sorted(val.specialty.unique().tolist())
except Exception:
    labels = [
        "Cardiovascular / Pulmonary", "Orthopedic", "Gastroenterology", "Neurology",
        "Obstetrics / Gynecology", "Urology", "ENT - Otolaryngology", "Nephrology",
        "Dermatology"
    ]

# Define the fallback label for cases where the API call or parsing fails
FALLBACK_LABEL = "Cardiovascular / Pulmonary"

# Only prompt if the key isn't already set
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

RUN_LLM = bool(os.environ.get("ANTHROPIC_API_KEY"))

if not RUN_LLM:
    print("ANTHROPIC_API_KEY not set - skipping live call.")
else:
    from anthropic import Anthropic
    client = Anthropic()
    model_id = "claude-sonnet-5-5"
    prompt_version = "zero_shot_v1"

    schema = {
        "type": "object",
        "properties": {"specialty": {"type": "string", "enum": labels}},
        "required": ["specialty"],
        "additionalProperties": False,
    }
    system_prompt = (
        "Classify a de-identified clinical note for a nonemergency outpatient "
        "referral. Choose exactly one specialty from the allowed list. "
        "Use only the supplied pre-decision text. Do not infer an answer from "
        "missing assessment, diagnosis, procedure, or referral sections. "
        "This is a research classification task, not a clinical decision."
    )

    results = []
    for row in val.itertuples(index=False):
        prediction = None
        try:
            response = client.messages.create(
                model=model_id,
                max_tokens=128,
                system=system_prompt,
                messages=[{
                    "role": "user",
                    "content": f"Allowed specialties: {', '.join(labels)}\n\nPre-decision note:\n{row.text}",
                }],
                output_config={"format": {"type": "json_schema", "schema": schema}},
            )
            block = next(b for b in response.content if b.type == "text")
            parsed = json.loads(block.text)
            prediction = parsed.get("specialty", FALLBACK_LABEL)
        except StopIteration:
            print(f"Warning: StopIteration for {row.case_id}. No text block found. Using fallback.")
            prediction = FALLBACK_LABEL
        except APIError as e:
            print(f"Warning: Anthropic APIError for {row.case_id}: {e}. Using fallback.")
            prediction = FALLBACK_LABEL
        except Exception as e:
            print(f"Warning: Parsing/Unexpected error for {row.case_id}: {e}. Using fallback.")
            prediction = FALLBACK_LABEL

        results.append({
            "case_id": row.case_id,
            "gold": row.specialty,
            "prediction": prediction,
            "model_id": model_id,
            "prompt_version": prompt_version
        })

    out = pd.DataFrame(results)
    out.to_csv("claude_validation_predictions.csv", index=False)

    print("Validation cases:", len(val))
    print("Claude predictions:", len(out))

    assert out.case_id.is_unique
    assert len(out) == len(val)
    assert set(out.case_id) == set(val.case_id)
    assert out.prediction.isin(labels).all()

    try:
        report("Claude (zero-shot)", out.gold, out.prediction)
    except NameError:
        from sklearn.metrics import accuracy_score, f1_score, classification_report
        acc = accuracy_score(out.gold, out.prediction)
        f1 = f1_score(out.gold, out.prediction, labels=labels, average="macro", zero_division=0)
        print(f"\n=== Claude (zero-shot) ===\nTop-1 accuracy: {acc:.3f}   Macro F1: {f1:.3f}")
        print(classification_report(out.gold, out.prediction, labels=labels, zero_division=0))

Validation cases: 126
Claude predictions: 126

=== Claude (zero-shot) ===
Top-1 accuracy: 0.889   Macro F1: 0.869
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.95      0.95      0.95        39
               Dermatology       1.00      1.00      1.00         3
      ENT - Otolaryngology       0.83      0.83      0.83         6
          Gastroenterology       0.80      0.92      0.86        13
                Nephrology       0.71      0.56      0.62         9
                 Neurology       0.75      1.00      0.86         9
   Obstetrics / Gynecology       0.92      1.00      0.96        12
                Orthopedic       0.95      0.84      0.89        25
                   Urology       0.89      0.80      0.84        10

                  accuracy                           0.89       126
                 macro avg       0.87      0.88      0.87       126
              weighted avg       0.89      0.89      0.89       126


## 13. Per-class recall with raw counts

Percentages alone hide sample size on small classes (Dermatology has 3 validation
cases, Nephrology 9). Report both.

In [ ]:
from sklearn.metrics import recall_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Ensure TF-IDF feature spaces are defined
try:
    _ = Xtr
    _ = Xval
except NameError:
    print("TF-IDF features are missing from memory. Re-vectorizing text datasets...")
    tfidf_builder = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english")
    Xtr = tfidf_builder.fit_transform(train["text"])
    Xval = tfidf_builder.transform(val["text"])

# Ensure mlp_64 is defined and fitted properly
from sklearn.neural_network import MLPClassifier
try:
    is_fitted = hasattr(mlp_64, "classes_")
except NameError:
    is_fitted = False

if not is_fitted:
    print("Fitting MLP 64 model...")
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mlp_64 = MLPClassifier(hidden_layer_sizes=(64,), max_iter=3000,
                               early_stopping=False, random_state=SEED, alpha=1e-3)
        mlp_64.fit(Xtr, train["specialty"])

final_model_pred = mlp_64.predict(Xval)  # swap in whichever candidate is selected
for cls in labels:
    n = (val["specialty"] == cls).sum()
    correct = ((val["specialty"] == cls) & (pd.Series(final_model_pred, index=val.index) == cls)).sum()
    print(f"{cls:30s}: {correct}/{n} correct")

## 14. Interpretability — global view + occlusion

Top predictive terms per class (global), plus one occlusion example on a correctly
classified case (removing that case's top-weighted words and observing the probability
drop for the true class).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Ensure tfidf vectorizer is defined and fitted
try:
    _ = tfidf
except NameError:
    print("tfidf is not defined. Re-fitting TfidfVectorizer...")
    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english")
    Xtr = tfidf.fit_transform(train["text"])
    Xval = tfidf.transform(val["text"])

# Ensure logreg is defined and fitted
try:
    _ = logreg
    is_lr_fitted = hasattr(logreg, "classes_")
except NameError:
    is_lr_fitted = False

if not is_lr_fitted:
    print("logreg is not defined or not fitted. Re-fitting Logistic Regression...")
    logreg = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, train["specialty"])

feat_names = np.array(tfidf.get_feature_names_out())
print("=== Top 6 terms per class (Logistic Regression coefficients) ===")
for i, cls in enumerate(logreg.classes_):
    top_idx = np.argsort(logreg.coef_[i])[-6:][::-1]
    print(f"{cls:30s}: {', '.join(feat_names[top_idx])}")

pred_lr = logreg.predict(Xval)
correct_mask = pred_lr == val["specialty"].values
idx = int(np.argmax(correct_mask))
row = val.iloc[idx]
true_cls = row["specialty"]
cls_idx = list(logreg.classes_).index(true_cls)
base_proba = logreg.predict_proba(Xval[idx])[0][cls_idx]

top_words = feat_names[np.argsort(logreg.coef_[cls_idx])[-5:]]
occluded_text = str(row["text"])
for w in top_words:
    occluded_text = occluded_text.replace(w, "")
Xocc = tfidf.transform([occluded_text])
occ_proba = logreg.predict_proba(Xocc)[0][cls_idx]

print(f"\nCase {row['case_id']}  true={true_cls}")
print(f"P(true class) before occlusion: {base_proba:.3f}")
print(f"Top words removed: {list(top_words)}")
print(f"P(true class) after occlusion:  {occ_proba:.3f}  (drop of {base_proba - occ_proba:.3f})")

## 15. Summary table

In [18]:
import pandas as pd
import warnings
from sklearn.dummy import DummyClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

# Fallback for SEED if missing
try:
    _ = SEED
except NameError:
    SEED = 42

# Fallback to load datasets from CSV if missing from global memory
try:
    _ = train
    _ = val
except NameError:
    print("Loading split files 'train.csv' and 'validation.csv' from disk...")
    train = pd.read_csv("train.csv")
    val = pd.read_csv("validation.csv")

# Fallback to initialize labels list if missing
try:
    _ = labels
except NameError:
    labels = sorted(train["specialty"].unique().tolist())

# Ensure TF-IDF feature spaces are defined
try:
    _ = Xtr
    _ = Xval
except NameError:
    print("Re-building TF-IDF features...")
    tfidf_builder = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english")
    Xtr = tfidf_builder.fit_transform(train["text"])
    Xval = tfidf_builder.transform(val["text"])

# Ensure dummy baseline is defined and fitted
try:
    _ = dummy
except NameError:
    print("Re-fitting Majority Class Dummy baseline...")
    dummy = DummyClassifier(strategy="most_frequent").fit(train["text"], train["specialty"])

# Ensure logreg is defined and fitted
try:
    _ = logreg
except NameError:
    from sklearn.linear_model import LogisticRegression
    print("Re-fitting Logistic Regression (TF-IDF)...")
    logreg = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, train["specialty"])

# Ensure mlp_64 is defined and fitted
try:
    is_mlp64_fitted = hasattr(mlp_64, "classes_")
except NameError:
    is_mlp64_fitted = False
if not is_mlp64_fitted:
    print("Re-fitting MLP (64)...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mlp_64 = MLPClassifier(hidden_layer_sizes=(64,), max_iter=3000,
                               early_stopping=False, random_state=SEED, alpha=1e-3)
        mlp_64.fit(Xtr, train["specialty"])

# Ensure mlp_64_32 is defined and fitted
try:
    is_mlp32_fitted = hasattr(mlp_64_32, "classes_")
except NameError:
    is_mlp32_fitted = False
if not is_mlp32_fitted:
    print("Re-fitting MLP (64-32)...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mlp_64_32 = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=3000,
                                  early_stopping=False, random_state=SEED, alpha=1e-3)
        mlp_64_32.fit(Xtr, train["specialty"])

# Ensure keyword baseline predictions are available
try:
    _ = kw_pred
except NameError:
    print("Regenerating keyword predictions...")
    KEYWORDS = {
        "Cardiovascular / Pulmonary": ["heart", "cardiac", "coronary", "pulmonary", "lung", "chest pain", "ejection fraction", "artery"],
        "Orthopedic": ["fracture", "joint", "knee", "hip", "shoulder", "orthopedic", "bone", "ligament"],
        "Gastroenterology": ["abdomen", "abdominal", "colon", "bowel", "gastric", "liver", "gi ", "stomach"],
        "Obstetrics / Gynecology": ["pregnan", "uterus", "cervix", "ovary", "vaginal", "gestational", "fetal"],
        "Urology": ["bladder", "prostate", "urinary", "kidney stone", "urology", "urethra", "scrotal"],
        "Nephrology": ["renal", "kidney", "dialysis", "nephro", "creatinine", "glomerular"],
        "Neurology": ["seizure", "neurolog", "headache", "brain", "cranial", "stroke", "weakness"],
        "ENT - Otolaryngology": ["throat", "ear", "nasal", "sinus", "tonsil", "hearing", "larynx"],
        "Dermatology": ["skin", "rash", "lesion", "dermat", "mole", "biopsy of the skin"],
    }
    def keyword_predict(text):
        t = str(text).lower()
        scores = {c: sum(t.count(k) for k in kws) for c, kws in KEYWORDS.items()}
        best = max(scores, key=scores.get)
        return best if scores[best] > 0 else "Cardiovascular / Pulmonary"
    kw_pred = val["text"].apply(keyword_predict)

summary_rows = []
def add_row(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    summary_rows.append({"model": name, "top1_accuracy": round(acc,3), "macro_f1": round(f1,3)})

add_row("Majority class", val["specialty"], dummy.predict(val["text"]))
add_row("Keyword baseline", val["specialty"], kw_pred)
add_row("Logistic Regression (TF-IDF)", val["specialty"], logreg.predict(Xval))
add_row("MLP (64)", val["specialty"], mlp_64.predict(Xval))
add_row("MLP (64-32)", val["specialty"], mlp_64_32.predict(Xval))

try:
    if RUN_LLM:
        add_row("Claude (zero-shot)", out.gold, out.prediction)
except NameError:
    pass

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
summary.to_csv("week7_summary.csv", index=False)

                       model  top1_accuracy  macro_f1
              Majority class          0.310     0.053
            Keyword baseline          0.579     0.540
Logistic Regression (TF-IDF)          0.849     0.843
                    MLP (64)          0.849     0.817
                 MLP (64-32)          0.817     0.781
          Claude (zero-shot)          0.889     0.869


 **Intrepret**

### Interpretation & Selection Rationale

Based on our final validation audit, we observe a clear progression in model performance across our baselines, statistical comparators, and LLM candidates:

1. **Baselines:** The **Majority Class** benchmark yields a low baseline of **31.0% accuracy (0.053 Macro F1)**, while the handcrafted **Keyword Baseline** reaches **57.9% accuracy (0.540 Macro F1)**.
2. **Statistical Comparators:** Our TF-IDF-based models demonstrate substantial improvements, showing that n-gram representations capture complex clinical context much better than keyword matches. **Logistic Regression (TF-IDF)** achieves **84.9% accuracy (0.843 Macro F1)**, outperforming the **MLP (64-32)** variant (**81.7% accuracy, 0.781 Macro F1**) and matching **MLP (64)** in raw accuracy.
3. **LLM Candidate:** The zero-shot **Claude (sonnet-5-5)** model achieves our peak performance of **88.9% accuracy and an 0.869 Macro F1 score**, outperforming all statistical and neural network candidates on the pre-decision-truncated texts.

### Selected Model
We select the **Claude (zero-shot)** candidate as our production model because it achieves the highest validation performance (both in top-1 accuracy and macro F1 score) while maintaining strict adherence to our group-aware, split-leakage constraints.